<a href="https://colab.research.google.com/github/futabarentarou/Rentarou-interacting-with-api-python-project-tutorial/blob/main/02_XGboost_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![Descripción de la imagen](https://ichef.bbci.co.uk/ace/ws/800/cpsprodpb/72E3/production/_93111492_dbreuters1.jpg.webp)

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, f1_score, confusion_matrix,roc_curve
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import shap
import seaborn as sns

In [ ]:
#path_data_input="C:/Users/cliente145/Documents/4geeks/20.Boosting_Tree_REPASO/data/"

In [ ]:
# Cargar datos
df_credit = pd.read_csv("german_credit_data.csv")
del df_credit["Unnamed: 0"]

FileNotFoundError: [Errno 2] No such file or directory: 'german_credit_data.csv'

In [ ]:
df_credit.head()

In [ ]:
len(df_credit)

In [ ]:
df_credit.shape

In [ ]:
#df_credit['Risk'] = (df_credit['Risk'] == 'bad').astype(int)

In [ ]:
dict_risk={"good":0, "bad":1}

df_credit['Risk'] =df_credit['Risk'].map(dict_risk)

In [ ]:
df_credit['Risk'].value_counts()

In [ ]:
700/(700+300)

In [ ]:
300/(700+300)

In [ ]:
df_credit['Risk'].value_counts(normalize=True)

In [ ]:
#Risk
#0    700
#1    300

In [ ]:
df_credit.info()

In [ ]:
#Variables Numericas.
df_credit.describe()

In [ ]:
df_credit["Saving accounts"].value_counts()

In [ ]:
df_credit["Sex"].value_counts()

In [ ]:
df_credit_encoded = pd.get_dummies(df_credit.drop(columns=['Risk'])).astype(int)

In [ ]:
df_credit_encoded

In [ ]:
list(df_credit_encoded.columns)

In [ ]:
len(df_credit_encoded)

In [ ]:
X_credit = df_credit_encoded
y_credit = df_credit['Risk']

In [ ]:
X_train_cr, X_test_cr, y_train_cr, y_test_cr = train_test_split(X_credit, y_credit,stratify=y_credit ,test_size=0.2, random_state=42)


In [ ]:
y_train_cr.value_counts(normalize=True)

In [ ]:
y_test_cr.value_counts(normalize=True)

In [ ]:
# GridSearchCV para hiperparámetros
param_grid = {
    'n_estimators': [100, 200,500,1000],
    'max_depth': [6,8,10,12],
    'learning_rate': [0.001, 0.1],
    'subsample': [0.8, 1.0]
}
gs = GridSearchCV(estimator=XGBClassifier(random_state=42),
                  param_grid=param_grid, scoring='roc_auc', cv=5, verbose=1)
gs.fit(X_train_cr, y_train_cr)


In [ ]:
gs.best_estimator_

In [ ]:
credit_model = gs.best_estimator_
y_proba_cr = credit_model.predict_proba(X_test_cr)[:, 1]

In [ ]:
y_proba_cr

In [ ]:
# Evaluación por percentiles
percentiles = np.arange(0.1, 1.0, 0.05)
best_f1 = 0
best_threshold = 0
f1_scores = []
thresh_values = []

In [ ]:

print("\n💳 Evaluación por percentiles (German Credit):")
for p in percentiles:
    print(p)
    threshold = np.percentile(y_proba_cr, p * 100)
    print(threshold)
    print("------------------------------------------")
    y_pred_thresh = (y_proba_cr >= threshold).astype(int)
    auc = roc_auc_score(y_test_cr, y_proba_cr)
    f1 = f1_score(y_test_cr, y_pred_thresh)
    print(f"Percentil {int(p*100)} - Threshold: {threshold:.3f} | AUC: {auc:.3f} | F1: {f1:.3f}")
    f1_scores.append(f1)
    thresh_values.append(threshold)
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold


In [ ]:
# Calcular la curva ROC
from sklearn.metrics import roc_curve, auc

# Calcular la curva ROC
fpr, tpr, thresholds = roc_curve(y_test_cr, y_proba_cr)
roc_auc = auc(fpr, tpr)

# Graficar la curva
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f"ROC curve (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')  # Línea aleatoria
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Curva ROC - German Credit")
plt.legend(loc="lower right")
plt.grid(True)
plt.show()


In [ ]:
best_threshold

In [ ]:
# Reporte final con el mejor threshold
y_pred_best = (y_proba_cr >= best_threshold).astype(int)
print("\n✅ Mejor Threshold basado en F1:")
print(f"Threshold: {best_threshold:.3f} | F1 Score: {best_f1:.3f}")
print("\nClassification Report:")
print(classification_report(y_test_cr, y_pred_best))

In [ ]:
# Matriz de confusión
tn, fp, fn, tp = confusion_matrix(y_test_cr, y_pred_best).ravel()
sns.heatmap(confusion_matrix(y_test_cr, y_pred_best), annot=True, fmt='d', cmap='Blues')
plt.title("Matriz de Confusión (Mejor Threshold)")
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.show()

In [ ]:
# Curva F1 vs Threshold
plt.plot(thresh_values, f1_scores, marker='o')
plt.title("F1 Score vs Threshold")
plt.xlabel("Threshold")
plt.ylabel("F1 Score")
plt.grid(True)
plt.show()


In [ ]:
import shap

In [ ]:
"""
# SHAP para crédito
explainer_credit = shap.Explainer(credit_model, X_train_cr)
shap_values_credit = explainer_credit(X_test_cr)
shap.summary_plot(shap_values_credit, X_test_cr, show=False)
plt.tight_layout()
plt.show()
"""

In [ ]:
# Creación del explainer
explainer = shap.Explainer(credit_model.predict, X_test_cr)

# Calculo de valores SHAP
shap_values = explainer(X_test_cr)

# summary
shap.plots.beeswarm(shap_values)